In [1]:
import os
import sys
import torch
import argparse
import mlflow
import mlflow.pytorch

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


from data import *
from utils import *

from datetime import datetime

sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Disentanglement'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Accuracy'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Attack'))

In [2]:
parser = argparse.ArgumentParser(description='Gen-RKM Model')

parser.add_argument('--N', type=int, default=5_000, help='Total # of samples') # 2_000 -> 5_000
parser.add_argument('--mb_size', type=int, default=32, help='Mini-batch size. See utils.py') # 64 -> 32; maybe 48?
parser.add_argument('--h_dim', type=int, default=48, help='Dim of latent vector') # 48 -> 64; maybe 64
parser.add_argument('--capacity', type=int, default=48, help='Capacity of network. See utils.py')

parser.add_argument('--x_fdim', type=int, default=256, help='Input x_fdim. See utils.py')
parser.add_argument('--y_fdim', type=int, default=256, help='Input y_fdim. See utils.py')
parser.add_argument('--z_fdim', type=int, default=64, help='Input z_fdim. See utils.py')

parser.add_argument('--c_accu', type=float, default=50, help='Input weight on recons_error')
parser.add_argument('--start_iter', type=int, default=0, help='Input start_iter for training')

parser.add_argument('--lr', type=float, default=7e-05, help='Input learning rate for optimizer') # 1e-4 -> 7e-5 -> 5e-5
parser.add_argument('--max_epochs', type=int, default=500, help='Input max_epoch for cut-off') # 100 -> 300 -> 500
parser.add_argument('--device', type=str, default='cpu', help='Device type: cuda or cpu')
parser.add_argument('--workers', type=int, default=0, help='# of workers for dataloader')
parser.add_argument('--shuffle', type=bool, default=True, help='shuffle dataset: true or false')

opt, _ = parser.parse_known_args()

h_dim = opt.h_dim

## Loading Dataset & Attacks

In [3]:
xtrain, xtest, ipVec_dim, nChannels = get_mmcelebahq_dataloader(args=opt)

info 0 5000 1


/Users/arthurclarysse/Desktop/🎓/03 Masterproef CS/VUB-MT-Development/Models/GenRKM-MMCelebHQ-V3/data.py:31: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4483.)
  label = torch.tensor(label, dtype=torch.double).T


info 5001 10001 1


In [4]:
# ct = time.strftime("%Y%m%d-%H%M")
ct = '20260312-1413'
dirs = create_dirs(ct=ct)
dirs.create()

In [5]:
mlflow.end_run()
mlflow.set_tracking_uri("file:./mlruns")

experiment_name = "PA-GenRKM-MMCelebAHQ"
mlflow.set_experiment(experiment_name)

exp = mlflow.get_experiment_by_name(experiment_name)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", exp.name)
print("Experiment ID:", exp.experiment_id)

run_name = f"PA-GenRKM-{opt.max_epochs}-{opt.h_dim}-{opt.capacity}-{ct}"

Tracking URI: file:./mlruns
Experiment: PA-GenRKM-MMCelebAHQ
Experiment ID: 880281085659876805


/opt/anaconda3/envs/MT-Dev2/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [6]:
model_name = '3V'
max_batches = 32
i_val = 10
epsilon = 0.2

In [7]:
def kPCA(X, Y, Z, nviews=3, eps=1e-6):
    if nviews == 3:
        a = (X @ X.T + Y @ Y.T + Z @ Z.T) / nviews
    elif nviews == 2:
        a = (X @ X.T + Y @ Y.T) / nviews
    else:
        a = (X @ X.T) / nviews

    B = a.size(0)

    # Centering (same dtype/device as a)
    oneN = torch.ones(B, B, device=a.device, dtype=a.dtype) / B
    a = a - oneN @ a - a @ oneN + oneN @ a @ oneN

    # Force symmetry (numerical)
    a = 0.5 * (a + a.T)

    # Add small jitter to diagonal for conditioning
    a = a + eps * torch.eye(B, device=a.device, dtype=a.dtype)

    # Eigen-decomposition (ascending eigenvalues)
    evals, evecs = torch.linalg.eigh(a)

    # Sort descending
    idx = torch.argsort(evals, descending=True)
    evals = evals[idx]
    evecs = evecs[:, idx]

    # Return top components
    h = evecs[:, :h_dim]
    s = evals  # (B,)

    return h, s

In [8]:
# Energy function
def my_loss(output_im, data_im, output_sk, data_sk, output_la, data_la):
    h, s = kPCA(output_im, output_sk, output_la) # Latent representation and eigenvalues

    U = torch.mm(torch.t(output_im), h) # Get encoded data back from the latent representation
    V = torch.mm(torch.t(output_sk), h)
    W = torch.mm(torch.t(output_la), h)

    x_im_tilde = net_im_de(torch.mm(h, torch.t(U))) # Reconstruction of the input
    x_sk_tilde = net_sk_de(torch.mm(h, torch.t(V)))
    x_la_tilde = net_la_de(torch.mm(h, torch.t(W)))

    # cost
    f1 = torch.trace(torch.mm(torch.mm(output_im, U), torch.t(h))) + torch.trace(torch.mm(torch.mm(output_sk, V), torch.t(h))) + torch.trace(torch.mm(torch.mm(output_la, W), torch.t(h)))
    f2 = 0.5 * torch.trace(torch.mm(h, torch.mm(torch.diag(s[:h_dim]), torch.t(h))))
    f3 = 0.5 * ((torch.trace(torch.mm(torch.t(U), U))) + (torch.trace(torch.mm(torch.t(V), V))) + (torch.trace(torch.mm(torch.t(W), W))))

    recon_loss1 = torch.nn.MSELoss()
    recon_loss2 = torch.nn.MSELoss()
    recon_loss3 = torch.nn.BCEWithLogitsLoss()
    f4 = recon_loss1(x_im_tilde.view(-1, 49152), data_im.view(-1, 49152)) + recon_loss2(x_sk_tilde.view(-1, 16384), data_sk.view(-1, 16384)) + recon_loss3(x_la_tilde.view(-1, 40), data_la.view(-1, 40))  # reconstruction loss

    loss = - f1 + f3 + f2 + 0.5 * (- f1 + f3 + f2) ** 2 + 100 * f4
    return loss

In [9]:
net_im_en = NetImEn(opt).double().to(opt.device)
net_sk_en = NetSkEn(opt).double().to(opt.device)
net_la_en = NetLaEn(opt).double().to(opt.device)

net_im_de = NetImDe(opt).double().to(opt.device)
net_sk_de = NetSkDe(opt).double().to(opt.device)
net_la_de = NetLaDe(opt).double().to(opt.device)

In [10]:
params = list(net_im_en.parameters()) + list(net_sk_en.parameters()) + list(net_la_en.parameters()) + list(net_im_de.parameters()) + list(net_sk_de.parameters()) + list(net_la_de.parameters())

optimizer = torch.optim.AdamW(params, lr=opt.lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    threshold=1e-3,
    cooldown=0,
    min_lr=1e-6,
)

l_cost = 6  # Costs from where checkpoints will be saved
t = 1
cost = np.inf  # Initialize cost

losses = []

In [ ]:
global_step = 0
run_id = 0

with mlflow.start_run(experiment_id=exp.experiment_id, run_name=run_name) as run:
    run_id = run.info.run_id
    
    mlflow.log_params({
        "N": opt.N,
        "mb_size": opt.mb_size,
        "h_dim": opt.h_dim,
        "capacity": opt.capacity,
        "x_fdim": opt.x_fdim,
        "y_fdim": opt.y_fdim,
        "z_fdim": opt.z_fdim,
        "c_accu": opt.c_accu,
        "start_iter": opt.start_iter,
        "lr": opt.lr,
        "max_epochs": opt.max_epochs,
        "device": opt.device,
        "workers": opt.workers,
        "shuffle": opt.shuffle,
    })

    while t <= opt.max_epochs:
        avg_loss = 0

        for i, (data_im, data_sk, data_la) in enumerate(xtrain):
            data_sk = data_sk.to(opt.device)
            data_la = data_la.to(opt.device)

            # Create FGSM Batch
            data_im_attack = data_im.clone().detach().to(opt.device)
            data_im_attack.requires_grad_(True)

            optimizer.zero_grad()

            output_im = net_im_en(data_im_attack)
            output_sk = net_sk_en(data_sk)
            output_la = net_la_en(data_la)

            loss_attack = my_loss(output_im, data_im_attack, output_sk, data_sk, output_la, data_la)

            loss_attack.backward()

            adv_data_im = data_im_attack + epsilon * data_im_attack.grad.sign()
            adv_data_im = torch.clamp(adv_data_im, 0, 1).detach()

            # 2. Train on the adversarial batch
            optimizer.zero_grad()

            output_im_adv = net_im_en(adv_data_im)

            # Recompute these for the real training graph
            output_sk_adv = net_sk_en(data_sk)
            output_la_adv = net_la_en(data_la)

            loss_adv = my_loss(output_im_adv, adv_data_im, output_sk_adv, data_sk, output_la_adv, data_la)

            loss_adv.backward()

            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()

            mlflow.log_metric("train_batch_loss", loss_adv.item(), step=global_step)
            mlflow.log_metric("attack_batch_loss", loss_attack.item(), step=global_step)
            mlflow.log_metric("learning_rate", optimizer.param_groups[0]["lr"], step=global_step)

            global_step += 1
            avg_loss += loss_adv.detach().cpu().item()

        cost = avg_loss / len(xtrain)
        scheduler.step(cost)

        mlflow.log_metric("train_epoch_loss", float(cost), step=t)
        t += 1

In [ ]:
# Get latest model
if os.path.exists('cp/{}'.format(dirs.dircp)):
    sd_mdl = torch.load('cp/{}'.format(dirs.dircp), weights_only=False)
    net_im_en.load_state_dict(sd_mdl['net_im_en_state_dict'])
    net_sk_en.load_state_dict(sd_mdl['net_sk_en_state_dict'])
    net_la_en.load_state_dict(sd_mdl['net_la_en_state_dict'])

    net_im_de.load_state_dict(sd_mdl['net_im_de_state_dict'])
    net_sk_de.load_state_dict(sd_mdl['net_sk_de_state_dict'])
    net_la_de.load_state_dict(sd_mdl['net_la_de_state_dict'])

U, V, W, h, s = mmcelebahq_final_compute(opt, net_im_en, net_sk_en, net_la_en, kPCA)

info 5001 10001 1
info 5001 10001 1


In [ ]:
# Save Model
torch.save({'net_im_en': net_im_en,
            'net_sk_en': net_sk_en,
            'net_la_en': net_la_en,

            'net_im_de': net_im_de,
            'net_sk_de': net_sk_de,
            'net_la_de': net_la_de,


            'net_im_en_state_dict': net_im_en.state_dict(),
            'net_sk_en_state_dict': net_sk_en.state_dict(),
            'net_la_en_state_dict': net_la_en.state_dict(),

            'net_im_de_state_dict': net_im_de.state_dict(),
            'net_sk_de_state_dict': net_sk_de.state_dict(),
            'net_la_de_state_dict': net_la_de.state_dict(),

            'data_im': data_im,
            'data_sk': data_sk,
            'data_la': data_la,
            'classes': get_classes()
        }, 'out/{}'.format(dirs.dirout))

In [ ]:
model = torch.load(f'./out/Attack-{model_name}.tar', weights_only=False)

data_im = model['data_im']
data_sk = model['data_sk']
data_la = model['data_la']

net_im_en = model['net_im_en']
net_sk_en = model['net_sk_en']
net_la_en = model['net_la_en']

net_im_de = model['net_im_de']
net_sk_de = model['net_sk_de']
net_la_de = model['net_la_de']